# ULTRON Unlimited Free Video Generator Server (Google Colab GPU)

This notebook allows you to turn a **Google Colab Free GPU (Tesla T4)** into a private video-generation server for your ULTRON chatbot! This allows you to generate completely free, unlimited AI videos without any expensive API keys.

### Instructions:
1. **Set GPU runtime**: In the menu bar above, go to **Runtime > Change runtime type**, select **T4 GPU** (or any higher GPU), and click **Save**.
2. **Get Ngrok Token**: Go to [ngrok.com](https://ngrok.com/) (free registration) and copy your **Authtoken** from the dashboard.
3. **Run Cell 1**: Installs dependencies (PyTorch, Diffusers, Flask, Ngrok).
4. **Run Cell 2**: Prompts for your Ngrok token, establishes a secure tunnel, loads the AI model, and launches the server.
5. **Copy Tunnel URL**: Once the server starts, copy the Ngrok tunnel URL (e.g. `https://xxxx-xx-xx.ngrok-free.app`) and paste it as `COLAB_VIDEO_URL` in your `.env.local` or Vercel environment settings!

In [ ]:
# Cell 1: Install required packages
!pip install -q diffusers transformers accelerate torch Flask flask-cors pyngrok

In [ ]:
# Cell 2: Setup secure tunnel and start Flask server
import os
import torch
import tempfile
from flask import Flask, request, send_file, jsonify
from flask_cors import CORS
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)

# 1. Input Ngrok Auth Token
print("=== NGROK TUNNEL SETUP ===")
NGROK_TOKEN = input("Paste your Ngrok Authtoken here: ").strip()
if not NGROK_TOKEN:
    raise ValueError("Ngrok token is required!")
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Load Damo Vilab Text-to-Video 1.7B Model
print("\n=== LOADING TEXT-TO-VIDEO MODEL ===")
print("This might take 1-2 minutes to download on first run...")
pipe = DiffusionPipeline.from_pretrained(
    "damo-vilab/text-to-video-ms-1.7b", 
    torch_dtype=torch.float16, 
    variant="fp16"
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
print("Model loaded successfully on GPU!")

@app.route('/generate', methods=['POST'])
def generate_video():
    try:
        data = request.json or {}
        prompt = data.get('prompt', '')
        if not prompt:
            return jsonify({'error': 'No prompt provided'}), 400

        print(f"\nGenerating video for prompt: '{prompt}'...")
        
        # Generate 16 frames of video (about 2 seconds, 256x256 resolution for T4 speed & memory safety)
        video_frames = pipe(prompt, num_inference_steps=25, num_frames=16).frames[0]
        
        # Save generated frames to a temporary MP4 file
        with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as tmp:
            video_path = export_to_video(video_frames, output_video_path=tmp.name, fps=8)
            print(f"Success! Sending video file.")
            return send_file(video_path, mimetype='video/mp4')
            
    except Exception as e:
        print("Error during video generation:", e)
        return jsonify({'error': str(e)}), 500

# 3. Open Tunnel and start Flask
public_url = ngrok.connect(5000)
print("\n" + "=" * 60)
print(f"COPY THIS TUNNEL URL:")
print(f"{public_url.public_url}")
print(f"\nPASTE IT IN YOUR .env.local AS:")
print(f'COLAB_VIDEO_URL="{public_url.public_url}"')
print("=" * 60 + "\n")

app.run(port=5000)